# Prototype clear sar
Clear and filter sentinel data

In [15]:
import pandas as pd
import os 
import pyarrow.parquet as pq

# Set

In [ ]:
year = 2024
raw_path = '../data/raw'
ds_name = f"s2_vessel_detections_{str(year)}.parquet"

inp = os.path.join(raw_path, ds_name)

out_path = '../data/interim'
out_name = f'cleaned_{ds_name}'
out = os.path.join(out_path, out_name)

# Import 

In [48]:
# pq.read_table -> get parquet data with the proper lib
# use_pandas_metadata=False -> ignore the pandas metadata
table = pq.read_table(inp, use_pandas_metadata=False)  
# to_pandas -> convert to pandas dataframe
# ignore_metadata=True -> ignore the pandas metadata
df = table.to_pandas(ignore_metadata=True)  

# Clean
Drop, rename, sort columns.

In [49]:
cols_to_drop = [
    'detect_timestamp', 
] 

df = df.drop(columns=cols_to_drop)

In [50]:
rename_dict = {
    'detect_lat': 'latitude',
    'detect_lon': 'longitude',
}

df = df.rename(columns=rename_dict)

In [51]:
cols_ordered = [
    'date', 'latitude', 'longitude', 'ssvid', 'speed_kn_inferred', 'heading_deg_inferred', 'length_m_inferred','presence_score', 'matching_score', 'cloud_score', 'likely_infrastructure'
]

df = df[cols_ordered]

In [52]:
df.head()

,date,latitude,longitude,ssvid,speed_kn_inferred,heading_deg_inferred,length_m_inferred,presence_score,matching_score,cloud_score,likely_infrastructure
0,2024-01-13,45.773047,13.592698,247058480,4.19,128.915504,29.9,0.853,0.000049,0.072032,False
1,2024-01-13,45.555515,13.708710,None,10.29,134.645082,35.4,0.906,0.000000,0.038961,False
2,2024-01-13,45.518864,13.564095,368345260,11.54,182.291831,28.6,0.906,0.000224,0.013158,False
3,2024-01-13,45.500175,13.496731,278001375,1.09,45.263666,24.9,0.804,0.000390,0.000761,False
4,2024-01-13,45.759770,13.596247,None,2.99,320.374660,33.8,0.868,0.000000,0.000962,False


# Filter fishing vessels

I chose the following filter:
- speed_kn_inferred: [0,9] (vessel ship at most in [2,8], and some at [0,8])
- lengh_m_inferred: <25 (usually at a maximum length of 18 m)
- presence_score: > 0.5
- cloud score: < 0.5 
- likely_infrascture: False 

In [53]:
n_total = len(df)

# Filters
spd_mask = df['speed_kn_inferred'] < 9
len_mask = df['length_m_inferred'] < 25
prs_mask = df['presence_score'] > 0.5
cld_mask = df['cloud_score'] < 0.5
inf_mask = df['likely_infrastructure'] == False

# Show how many elements are removed for each filter 
masks = {
    "speed < 9": spd_mask,
    "length < 25": len_mask,
    "presence > 0.5": prs_mask,
    "cloud < 0.5": cld_mask,
    "not infrastructure": inf_mask,
}

for name, mask in masks.items():
    n_pass = mask.sum()
    n_removed = n_total - n_pass
    perc_removed = 100 * n_removed / n_total
    print(f"{name:25s} removed {n_removed:8d} rows ({perc_removed:6.2f}%)")

# Perform the filter 
full_mask = spd_mask & len_mask & prs_mask & cld_mask & inf_mask
n_after = full_mask.sum()
print("\nCombined filters:")
print(f"  Records before filtering: {n_total}")
print(f"  Records after filtering:  {n_after}")
print(f"  Records removed:          {n_total - n_after} ({100*(n_total-n_after)/n_total:.2f}%)")

df = df.loc[full_mask].copy()


speed < 9                 removed       24 rows (  6.82%)
length < 25               removed      229 rows ( 65.06%)
presence > 0.5            removed       33 rows (  9.38%)
cloud < 0.5               removed       32 rows (  9.09%)
not infrastructure        removed        0 rows (  0.00%)

Combined filters:
  Records before filtering: 352
  Records after filtering:  100
  Records removed:          252 (71.59%)


# Clean no more useful columns 

In [55]:
cols_to_drop = [
    'matching_score',
    'cloud_score',
    'likely_infrastructure',
]

df = df.drop(columns=cols_to_drop)

# Export 

In [58]:
df.to_parquet(out, index=False)